## **Weeks 2–3 – Dataset Structuring and Validation & Mortgage Rate Enrichment**

### Part 1: Dataset Structuring and Validation

**Objective:**  Inspect and filter the datasets to ensure only relevant residential property records are used. This week also covers foundational EDA steps that inform every phase that
follows.

**Skills Learned:** Dataset validation and quality checks • EDA • MLS dataset structure and property type filtering

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

# Load combined datasets from intermediate folder
sold = pd.read_csv('../data/02_intermediate/CRMLSSold_Combined.csv', low_memory=False)
listings = pd.read_csv('../data/02_intermediate/CRMLSListing_Combined.csv', low_memory=False)

### Dataset Understanding
1. Identify number of rows and columns, date range
2. Review column data types
3. Identify high-missing columns
4. Separate market analysis fields from metadata fields

In [4]:
print(f"Sold: {sold.shape[0]:,} rows, {sold.shape[1]} columns")
print(f"Listings: {listings.shape[0]:,} rows, {listings.shape[1]} columns")

Sold: 421,589 rows, 80 columns
Listings: 467,728 rows, 82 columns


In [5]:
print("Sold date range:", sold['CloseDate'].min(), "to", sold['CloseDate'].max())
print("Listings date range:", listings['ListingContractDate'].min(), "to", listings['ListingContractDate'].max())

Sold date range: 2024-01-01 to 2026-04-30
Listings date range: 2024-01-01 to 2026-04-30


In [7]:
print("SOLD column data types:")
print(sold.dtypes.to_string())

SOLD column data types:
BuyerAgentAOR                    object
ListAgentAOR                     object
Flooring                         object
ViewYN                           object
WaterfrontYN                     object
BasementYN                       object
PoolPrivateYN                    object
OriginalListPrice               float64
ListingKey                        int64
CloseDate                        object
ClosePrice                      float64
ListAgentFirstName               object
ListAgentLastName                object
Latitude                        float64
Longitude                       float64
UnparsedAddress                  object
PropertyType                     object
LivingArea                      float64
ListPrice                       float64
DaysOnMarket                      int64
ListOfficeName                   object
BuyerOfficeName                  object
CoListOfficeName                 object
ListAgentFullName                object
CoListAgentFirst

In [9]:
# Separate market analysis fields from metadata fields

market_fields = [
    # Prices
    'ClosePrice', 'ListPrice', 'OriginalListPrice',
    # Property details
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'YearBuilt',
    'LotSizeAcres', 'LotSizeSquareFeet', 'LotSizeArea', 'PropertySubType',
    # Time
    'CloseDate', 'ListingContractDate', 'PurchaseContractDate',
    'ContractStatusChangeDate', 'DaysOnMarket',
    # Location
    'City', 'CountyOrParish', 'PostalCode', 'StateOrProvince',
    'Latitude', 'Longitude', 'UnparsedAddress', 'MLSAreaMajor',
    # Property characteristics
    'PropertyType', 'MlsStatus', 'YearBuilt', 'GarageSpaces',
    'FireplaceYN', 'PoolPrivateYN', 'NewConstructionYN', 'Stories',
    'AssociationFee', 'AssociationFeeFrequency', 'ParkingTotal',
    # Agent & office (for competitive intelligence)
    'ListOfficeName', 'BuyerOfficeName', 'ListAgentFullName',
    'ListAgentFirstName', 'ListAgentLastName',
    'BuyerAgentFirstName', 'BuyerAgentLastName',
]

metadata_fields = [
    # System identifiers
    'ListingKey', 'ListingKeyNumeric', 'ListingId',
    # MLS system info
    'OriginatingSystemName', 'OriginatingSystemSubName',
    # Internal agent IDs
    'BuyerAgentMlsId', 'ListAgentAOR', 'BuyerAgentAOR', 'BuyerOfficeAOR',
    # Tax records (also 100% missing)
    'TaxYear', 'TaxAnnualAmount',
]

print(f"Market analysis fields: {len(market_fields)}")
print(f"Metadata fields: {len(metadata_fields)}")

Market analysis fields: 42
Metadata fields: 11


### Missing Value Analysis
5. Calculate missing counts and percentages per column
6. Flag columns with >90% missing values
7. Decide which columns to drop vs. retain (keep core fields even if partially missing)

In [13]:
# 5.  Calculate missing counts and percentages per column

missing = pd.DataFrame({
    'missing_count': sold.isnull().sum(),
    'missing_pct': (sold.isnull().sum() / len(sold) * 100).round(2)
})
missing = missing.sort_values('missing_pct', ascending=False)

In [12]:
# 6. Flag columns above 90% missing values.

high_missing = missing[missing['missing_pct'] > 90]
print(f"Columns with >90% missing values ({len(high_missing)} total):")
high_missing

Columns with >90% missing values (16 total):


,missing_count,missing_pct
FireplacesTotal,421589,100.00
MiddleOrJuniorSchoolDistrict,421589,100.00
CoveredSpaces,421589,100.00
BusinessType,421589,100.00
ElementarySchoolDistrict,421589,100.00
TaxYear,421589,100.00
TaxAnnualAmount,421589,100.00
AboveGradeFinishedArea,421589,100.00
WaterfrontYN,421325,99.94
BelowGradeFinishedArea,419126,99.42


In [17]:
# 7. Decide which columns to drop vs. retain.

# These columns are >90% missing and carry no analytical value

columns_to_drop = high_missing.index.tolist()
print(f"Columns flagged for dropping ({len(columns_to_drop)} total):")
print(columns_to_drop)

# Note: columns will not be dropped yet.

Columns flagged for dropping (16 total):
['FireplacesTotal', 'MiddleOrJuniorSchoolDistrict', 'CoveredSpaces', 'BusinessType', 'ElementarySchoolDistrict', 'TaxYear', 'TaxAnnualAmount', 'AboveGradeFinishedArea', 'WaterfrontYN', 'BelowGradeFinishedArea', 'BasementYN', 'ListAgentEmail', 'LotSizeDimensions', 'BuilderName', 'BuildingAreaTotal', 'CoBuyerAgentFirstName']


### Numeric Distribution Review
For the following key numeric fields (ClosePrice, ListPrice, OriginalListPrice, LivingArea, LotSizeAcres, BedroomsTotal BathroomsTotalInteger, DaysOnMarket, and YearBuilt) analyze its distribution by:

8. Generating a percentile summary
9. Creating Histograms & Boxplots
10. Identifying extreme outliers for later handling 

In [19]:
# 8. Generate a percentile summary
numeric_fields = ['ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea',
                  'LotSizeAcres', 'BedroomsTotal', 'BathroomsTotalInteger',
                  'DaysOnMarket', 'YearBuilt']

print(sold[numeric_fields].describe(percentiles=[.10, .25, .50, .75, .90, .95, .99]))

         ClosePrice     ListPrice  OriginalListPrice    LivingArea  \
count  4.215890e+05  4.215890e+05       4.207260e+05  4.213390e+05   
mean   1.126447e+06  1.124413e+06       1.206049e+06  1.899450e+03   
std    2.421839e+06  1.354276e+06       6.569939e+06  2.623946e+04   
min    5.250000e+02  5.250000e+02       0.000000e+00  0.000000e+00   
10%    4.150000e+05  4.150000e+05       4.249000e+05  9.800000e+02   
25%    5.700000e+05  5.750000e+05       5.800000e+05  1.245000e+03   
50%    8.125000e+05  7.999990e+05       8.180000e+05  1.640000e+03   
75%    1.275000e+06  1.265000e+06       1.290000e+06  2.215000e+03   
90%    2.000000e+06  1.995000e+06       1.999000e+06  2.970000e+03   
95%    2.788000e+06  2.795000e+06       2.850000e+06  3.549000e+03   
99%    5.430150e+06  5.600000e+06       5.950000e+06  5.258000e+03   
max    7.960000e+08  1.375000e+08       1.390000e+09  1.702132e+07   

       LotSizeAcres  BedroomsTotal  BathroomsTotalInteger   DaysOnMarket  \
count  3.8802

In [ ]:
# 9a. Create Histograms for all numeric fields

fig, axes = plt.subplots(3, 3, figsize=(12, 9))
fig.suptitle('Numeric Field Distributions - Sold Dataset', fontsize=16)

for i, field in enumerate(numeric_fields):
    ax = axes[i//3, i%3]
    sold[field].dropna().plot(kind='hist', bins=50, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(field)

plt.tight_layout()
plt.savefig('../results/sold_distributions.png')
plt.show()

In [ ]:
# 9b. Create Boxplots for all numeric fields

fig, axes = plt.subplots(3, 3, figsize=(12, 9))
fig.suptitle('Boxplots - Sold Dataset', fontsize=16)

for i, field in enumerate(numeric_fields):
    ax = axes[i//3, i%3]
    sold[field].dropna().plot(kind='box', ax=ax, color='steelblue')
    ax.set_title(field)

plt.tight_layout()
plt.savefig('../results/sold_boxplots.png')
plt.show()

In [22]:
# 10. Identify extreme outliers for later handling 

print("Extreme outliers in key numeric fields:")
for field in numeric_fields:
    Q1 = sold[field].quantile(0.25)
    Q3 = sold[field].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_count = ((sold[field] < lower) | (sold[field] > upper)).sum()
    print(f"{field:<25} {outlier_count:>7,} outliers | normal range {lower:>12,.0f} to {upper:>12,.0f}")

Extreme outliers in key numeric fields:
ClosePrice                 31,098 outliers | normal range     -487,500 to    2,332,500
ListPrice                  31,097 outliers | normal range     -460,000 to    2,300,000
OriginalListPrice          31,501 outliers | normal range     -485,000 to    2,355,000
LivingArea                 18,426 outliers | normal range         -210 to        3,670
LotSizeAcres               60,365 outliers | normal range           -0 to            1
BedroomsTotal              23,202 outliers | normal range            2 to            6
BathroomsTotalInteger      19,206 outliers | normal range            0 to            4
DaysOnMarket               31,651 outliers | normal range          -54 to          110
YearBuilt                     982 outliers | normal range        1,902 to        2,058


### Answering Suggested Intern Questions

In [28]:
# 1. What is the Residential vs other property type share?
sample = pd.read_csv('../data/01_raw/CRMLSSold202604.csv', low_memory=False)
print("Property type breakdown (sample from April 2026):")
print(sample['PropertyType'].value_counts())
print(f"\nResidential share: {sample['PropertyType'].value_counts(normalize=True).get('Residential', 0)*100:.1f}%")

# 2. What are the median and average close prices?
print(f"\nMedian close price: ${sold['ClosePrice'].median():,.0f}")
print(f"Average close price: ${sold['ClosePrice'].mean():,.0f}")

# 3. What percentage of homes sold above vs below list price?
sold['sold_above_list'] = sold['ClosePrice'] >= sold['ListPrice']
pct_above = sold['sold_above_list'].mean() * 100
print(f"\nHomes sold at or above list price: {pct_above:.1f}%")
print(f"Homes sold below list price: {100 - pct_above:.1f}%")

# 4. Which counties have the highest median prices?
print("\nTop 10 counties by median close price:")
print(sold.groupby('CountyOrParish')['ClosePrice'].median()
      .sort_values(ascending=False).head(10)
      .apply(lambda x: f"${x:,.0f}"))

# 5. What does the Days on Market distribution look like?
print(f"\nDays on Market median: {sold['DaysOnMarket'].median():.0f} days")
print(f"Days on Market average: {sold['DaysOnMarket'].mean():.1f} days")
print(f"Negative values (bad data): {(sold['DaysOnMarket'] < 0).sum():,}")
print(f"Over 365 days: {(sold['DaysOnMarket'] > 365).sum():,}")

# 6. Are there any apparent date consistency issues?
sold['CloseDate'] = pd.to_datetime(sold['CloseDate'])
sold['ListingContractDate'] = pd.to_datetime(sold['ListingContractDate'])
sold['PurchaseContractDate'] = pd.to_datetime(sold['PurchaseContractDate'])

print("Date consistency issues:")
print(f"  Close date before listing date: {(sold['CloseDate'] < sold['ListingContractDate']).sum():,}")
print(f"  Close date before purchase contract date: {(sold['CloseDate'] < sold['PurchaseContractDate']).sum():,}")

Property type breakdown (sample from April 2026):
PropertyType
Residential            16581
ResidentialLease        5453
Land                     706
ManufacturedInPark       631
ResidentialIncome        630
CommercialSale           136
CommercialLease          113
BusinessOpportunity       11
Name: count, dtype: int64

Residential share: 68.3%

Median close price: $812,500
Average close price: $1,126,447

Homes sold at or above list price: 57.2%
Homes sold below list price: 42.8%

Top 10 counties by median close price:
CountyOrParish
Del Norte        $6,742,500
San Mateo        $1,650,000
Santa Clara      $1,540,000
Santa Cruz       $1,185,000
Orange           $1,175,000
San Francisco    $1,175,000
Marin            $1,150,000
Alameda          $1,120,000
Alpine           $1,100,000
Mono             $1,030,000
Name: ClosePrice, dtype: object

Days on Market median: 19 days
Days on Market average: 37.7 days
Negative values (bad data): 45
Over 365 days: 800
Date consistency issues:
  Clos

### Part 2: Mortgage Rate Enrichment

**Objective:**  Fetch the FRED MORTGAGE30US series, resample it from weekly to monthly frequency, and merge it onto both combined datasets using a year-month key derived from transaction dates.

**Skills Learned:** Fetching live data from a public API (FRED) • Resampling time-series data from W to M • Creating join keys from datetime fields • Left merging external economic data onto a transaction dataset • Validating merge completeness with null checks

### Mortgage Rate Enrichment

In [ ]:
# Step 1 - Fetching FRED MORTGAGE30US  Series
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date']) #Note, this said "DATE" on the handbook
mortgage.columns = ['date', 'rate_30yr_fixed']

In [45]:
# Step 2 — Resample weekly rates to monthly averages
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean().reset_index()
)

In [46]:
# Step 3 – Create a matching year_month key on the MLS datasets

# Sold dataset — key off CloseDate
sold["year_month"] = pd.to_datetime(sold["CloseDate"]).dt.to_period("M")

# Listings dataset — key off ListingContractDate
listings["year_month"] = pd.to_datetime(
    listings["ListingContractDate"]).dt.to_period("M")

In [47]:
# Step 4 – Merge
sold_with_rates = sold.merge(mortgage_monthly, on="year_month", how="left")
listings_with_rates = listings.merge(mortgage_monthly, on="year_month", how="left")

In [48]:
# Step 5 – Validate the merge

# Check for any unmatched rows (rate should not be null)
print(sold_with_rates["rate_30yr_fixed"].isnull().sum())
print(listings_with_rates["rate_30yr_fixed"].isnull().sum())

0
0


In [49]:
# Preview
print(sold_with_rates[["CloseDate", "year_month", "ClosePrice",
"rate_30yr_fixed"]].head())

   CloseDate year_month  ClosePrice  rate_30yr_fixed
0 2024-01-18    2024-01   5000000.0           6.6425
1 2024-01-30    2024-01    858000.0           6.6425
2 2024-01-29    2024-01   1890500.0           6.6425
3 2024-01-02    2024-01   2100000.0           6.6425
4 2024-01-22    2024-01   1950000.0           6.6425
